# SAE-XCrash — NB06: SOTA Comparison
**Work Package WP6** | P12: CatBoost + Random Forest with Target Encoding

Load-and-continue notebook. Run **after** NB01_Data_Protocol_v2 and NB02_Modeling_Baselines_v2.
Adds two new baselines to Table 4 in response to Reviewer 2, Comment 4.
All outputs go to `logs/wp6/` and `figures/wp6/`.

### Steps
| Step | Description | Paper location |
|------|-------------|----------------|
| 1 | Environment & Paths | — |
| 2 | Load processed parquets + WP2 metadata | — |
| 3 | P12a: CatBoost (32 features, full training set) | Table 4 row |
| 4 | P12b: RF with Target Encoding (34 features, 1M subsample) | Table 4 row |
| 5 | Summary table + save results | logs/wp6/p12_sota_results.json |

**GPU:** T4 is sufficient. CatBoost runs in CPU mode; RF is CPU-only.  
**Runtime:** ~25–40 min total on T4.  
**Seed:** `SEED = 42`

---
## Step 1 — Environment & Paths

In [1]:
!pip install -q catboost scikit-learn pyarrow matplotlib numpy pandas

import os, json, time, warnings
from pathlib import Path
from datetime import datetime
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import catboost as cb
from sklearn.ensemble import RandomForestClassifier
from sklearn.isotonic import IsotonicRegression
from sklearn.metrics import average_precision_score, roc_auc_score, brier_score_loss
warnings.filterwarnings('ignore')

from google.colab import drive
drive.mount('/content/drive', force_remount=False)

PROJ_ROOT = Path('/content/drive/MyDrive/SAE-XCrash')

# ── Exact paths matching NB02 conventions ─────────────────────────────────────
WP2_META_PATH = PROJ_ROOT / 'logs' / 'wp2_meta.json'
PROC_DIR = PROJ_ROOT / 'data' / 'processed'
TRAIN_PARQUET = PROC_DIR / 'usa_train_processed.parquet'
VAL_PARQUET = PROC_DIR / 'usa_val_processed.parquet'
TEST_PARQUET = PROC_DIR / 'usa_test_processed.parquet'

# ── New versioned output dirs ─────────────────────────────────────────────────
LOGS_DIR = PROJ_ROOT / 'logs'    / 'wp6'
FIGS_DIR = PROJ_ROOT / 'figures' / 'wp6'
LOGS_DIR.mkdir(parents=True, exist_ok=True)
FIGS_DIR.mkdir(parents=True, exist_ok=True)

SEED = 42
DPI = 300
ECE_BINS = 15

print(f"PROJ_ROOT  : {PROJ_ROOT}")
print(f"Train data : {TRAIN_PARQUET}")
print(f"Outputs    : {LOGS_DIR}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 10.2 MB/s eta 0:00:00
Mounted at /content/drive
PROJ_ROOT  : /content/drive/MyDrive/SAE-XCrash
Train data : /content/drive/MyDrive/SAE-XCrash/data/processed/usa_train_processed.parquet
Outputs    : /content/drive/MyDrive/SAE-XCrash/logs/wp6


---
## Step 2 — Load Processed Parquets + WP2 Metadata

In [2]:
# ── ECE helper (matching NB03 implementation) ─────────────────────────────────
def ece_score(y_true, y_prob, n_bins=ECE_BINS):
    bins = np.linspace(0, 1, n_bins + 1)
    ece = 0.0
    n = len(y_true)
    for i in range(n_bins):
        mask = (y_prob >= bins[i]) & (y_prob < bins[i+1])
        if mask.sum() == 0:
            continue
        ece += mask.sum() / n * abs(y_true[mask].mean() - y_prob[mask].mean())
    return round(float(ece), 6)

# ── Load WP2 metadata (feature list + class weight) ───────────────────────────
print("Loading WP2 metadata ...")
with open(WP2_META_PATH) as f:
    wp2_meta = json.load(f)

FEAT_V1 = wp2_meta['feat_usa']           # original 32 features (same as XGBoost/LightGBM)
SPW = float(wp2_meta.get('spw', 36.0))
print(f"  V1 features : {len(FEAT_V1)} | SPW: {SPW:.1f}")

# ── Load all three splits ─────────────────────────────────────────────────────
print("\nLoading parquets (this may take 2-3 min) ...")
t0 = time.time()
train_df = pd.read_parquet(TRAIN_PARQUET)
val_df = pd.read_parquet(VAL_PARQUET)
test_df = pd.read_parquet(TEST_PARQUET)
print(f"  Train : {len(train_df):,} rows")
print(f"  Val   : {len(val_df):,} rows")
print(f"  Test  : {len(test_df):,} rows")
print(f"  Loaded in {time.time()-t0:.0f}s")

# ── Identify target-encoded features added by NB01_v2 ─────────────────────────
TE_FEATS = [f for f in ['s_street_te', 's_city_te'] if f in train_df.columns]
FEAT_TE = FEAT_V1 + TE_FEATS            # 34 features for RF-TE
print(f"\n  Target-encoded features available: {TE_FEATS}")
print(f"  FEAT_V1 (CatBoost) : {len(FEAT_V1)} features")
print(f"  FEAT_TE (RF-TE)    : {len(FEAT_TE)} features")

# ── Prepare numpy arrays ──────────────────────────────────────────────────────
EXCL = {'label','ID','Start_Time','datetime',
        'accident_index','accident_reference','collision_reference','row_id'}

y_train = train_df['label'].values
y_val = val_df['label'].values
y_test = test_df['label'].values
no_skill = float(y_test.mean())

# 32-feature arrays (CatBoost)
X_train_v1 = train_df[FEAT_V1].values.astype(np.float32)
X_val_v1 = val_df[FEAT_V1].values.astype(np.float32)
X_test_v1 = test_df[FEAT_V1].values.astype(np.float32)

# 34-feature arrays (RF-TE)
X_train_te = train_df[FEAT_TE].values.astype(np.float32)
X_val_te = val_df[FEAT_TE].values.astype(np.float32)
X_test_te = test_df[FEAT_TE].values.astype(np.float32)

print(f"\n  No-skill AUPRC baseline: {no_skill:.4f}")
print(f"  Severe rate (train)    : {y_train.mean()*100:.2f}%")
print("\nAll data loaded ✓")

Loading WP2 metadata ...
  V1 features : 32 | SPW: 36.0

Loading parquets (this may take 2-3 min) ...
  Train : 5,549,870 rows
  Val   : 1,268,806 rows
  Test  : 166,552 rows
  Loaded in 17s

  Target-encoded features available: ['s_street_te', 's_city_te']
  FEAT_V1 (CatBoost) : 32 features
  FEAT_TE (RF-TE)    : 34 features

  No-skill AUPRC baseline: 0.0291
  Severe rate (train)    : 2.71%

All data loaded ✓


---
## Step 3 — P12a: CatBoost (32 features, full training set)

In [4]:
# ─────────────────────────────────────────────────────────────────────────────
# P12a — CatBoost
# Same 32 features and SPW as XGBoost/LightGBM.
# Trained on full training set; calibrated on val set.
# task_type='CPU' for T4 compatibility.
# ─────────────────────────────────────────────────────────────────────────────
print("P12a — CatBoost")
print("=" * 60)

cat_pool_train = cb.Pool(X_train_v1, label=y_train)
cat_pool_val = cb.Pool(X_val_v1,   label=y_val)

cat_model = cb.CatBoostClassifier(
    iterations = 1000,
    learning_rate = 0.05,
    depth = 8,
    scale_pos_weight = SPW,
    eval_metric = 'PRAUC',
    early_stopping_rounds = 50,
    random_seed = SEED,
    task_type = 'GPU',   # change to 'GPU' if A100 is available
    verbose = 100,
)

t0 = time.time()
cat_model.fit(
    cat_pool_train,
    eval_set = cat_pool_val,
    use_best_model = True,
)
cat_train_time = time.time() - t0
print(f"\n  Best iteration : {cat_model.best_iteration_}")
print(f"  Training time  : {cat_train_time:.0f}s")

# ── Scores ────────────────────────────────────────────────────────────────────
p_cat_val = cat_model.predict_proba(X_val_v1)[:, 1]
p_cat_test = cat_model.predict_proba(X_test_v1)[:, 1]

# ── Calibration (isotonic on val — matching NB03 protocol) ────────────────────
iso_cat = IsotonicRegression(out_of_bounds='clip')
iso_cat.fit(p_cat_val, y_val)
p_cat_test_cal = np.clip(iso_cat.predict(p_cat_test), 0, 1)

# ── Metrics ───────────────────────────────────────────────────────────────────
cat_results = {
    'model'     : 'CatBoost',
    'features'  : f'V1 ({len(FEAT_V1)})',
    'AUROC'     : round(roc_auc_score(y_test, p_cat_test), 4),
    'AUPRC'     : round(average_precision_score(y_test, p_cat_test), 4),
    'ECE_uncal' : ece_score(y_test, p_cat_test),
    'ECE_cal'   : ece_score(y_test, p_cat_test_cal),
    'Brier_cal' : round(brier_score_loss(y_test, p_cat_test_cal), 6),
    'train_time': f'{cat_train_time:.0f}s',
}

print("\n  Results:")
for k, v in cat_results.items():
    print(f"    {k:<15}: {v}")
print("\nP12a complete ✓")

P12a — CatBoost


Default metric period is 5 because PRAUC is/are not implemented for GPU
Metric PRAUC is not implemented on GPU. Will use CPU for metric computation, this could significantly affect learning time


0:	learn: 0.7202448	test: 0.6671196	best: 0.6671196 (0)	total: 587ms	remaining: 9m 46s
100:	learn: 0.7946791	test: 0.7032087	best: 0.7032087 (100)	total: 40.1s	remaining: 5m 56s
200:	learn: 0.8082680	test: 0.7193946	best: 0.7193946 (200)	total: 1m 24s	remaining: 5m 35s
300:	learn: 0.8157761	test: 0.7280898	best: 0.7280898 (300)	total: 2m 9s	remaining: 5m 1s
400:	learn: 0.8211356	test: 0.7320373	best: 0.7320373 (400)	total: 2m 55s	remaining: 4m 21s
500:	learn: 0.8259627	test: 0.7357541	best: 0.7357541 (500)	total: 3m 40s	remaining: 3m 39s
600:	learn: 0.8295187	test: 0.7381467	best: 0.7381467 (600)	total: 4m 25s	remaining: 2m 56s
700:	learn: 0.8328560	test: 0.7399074	best: 0.7399162 (699)	total: 5m 10s	remaining: 2m 12s
800:	learn: 0.8357359	test: 0.7416850	best: 0.7416867 (797)	total: 5m 56s	remaining: 1m 28s
900:	learn: 0.8382566	test: 0.7424144	best: 0.7424507 (899)	total: 6m 41s	remaining: 44.2s
999:	learn: 0.8405984	test: 0.7426844	best: 0.7427132 (991)	total: 7m 27s	remaining: 0us


---
## Step 4 — P12b: Random Forest with Target Encoding (1M stratified subsample)

In [5]:
# ─────────────────────────────────────────────────────────────────────────────
# P12b — Random Forest with Target Encoding (RF-TE)
# Uses 34 features: 32 original + s_street_te + s_city_te (from NB01_v2).
# 1M stratified subsample used for training to avoid OOM on T4 (12GB RAM).
# RF is CPU-only (sklearn); n_jobs=-1 uses all available cores.
# ─────────────────────────────────────────────────────────────────────────────
print("P12b — Random Forest with Target Encoding (RF-TE)")
print("=" * 60)

# ── Stratified 1M subsample ───────────────────────────────────────────────────
RF_SAMPLE = 1_000_000
rng = np.random.default_rng(SEED)
idx_sev = np.where(y_train == 1)[0]
idx_non = np.where(y_train == 0)[0]
n_sev = int(RF_SAMPLE * y_train.mean())
n_non = RF_SAMPLE - n_sev
idx_sub = np.concatenate([
    rng.choice(idx_sev, min(n_sev, len(idx_sev)), replace=False),
    rng.choice(idx_non, n_non, replace=False),
])
rng.shuffle(idx_sub)
X_rf_sub = X_train_te[idx_sub]
y_rf_sub = y_train[idx_sub]
print(f"  RF subsample : {len(X_rf_sub):,} rows")
print(f"  Severe rate  : {y_rf_sub.mean()*100:.2f}%")
print(f"  Features     : {len(FEAT_TE)} ({FEAT_V1[:2]}...+ {TE_FEATS})")

# ── Train ─────────────────────────────────────────────────────────────────────
rf_model = RandomForestClassifier(
    n_estimators = 200,
    max_depth = 15,
    class_weight = 'balanced_subsample',
    max_features = 'sqrt',
    n_jobs = -1,
    random_state = SEED,
)

t0 = time.time()
rf_model.fit(X_rf_sub, y_rf_sub)
rf_train_time = time.time() - t0
print(f"\n  Training time : {rf_train_time:.0f}s")

# ── Scores ────────────────────────────────────────────────────────────────────
p_rf_val = rf_model.predict_proba(X_val_te)[:, 1]
p_rf_test = rf_model.predict_proba(X_test_te)[:, 1]

# ── Calibration (isotonic on val — matching NB03 protocol) ────────────────────
iso_rf = IsotonicRegression(out_of_bounds='clip')
iso_rf.fit(p_rf_val, y_val)
p_rf_test_cal = np.clip(iso_rf.predict(p_rf_test), 0, 1)

# ── Metrics ───────────────────────────────────────────────────────────────────
rf_results = {
    'model'     : 'RF-TE',
    'features'  : f'TE ({len(FEAT_TE)}, 1M subsample)',
    'AUROC'     : round(roc_auc_score(y_test, p_rf_test), 4),
    'AUPRC'     : round(average_precision_score(y_test, p_rf_test), 4),
    'ECE_uncal' : ece_score(y_test, p_rf_test),
    'ECE_cal'   : ece_score(y_test, p_rf_test_cal),
    'Brier_cal' : round(brier_score_loss(y_test, p_rf_test_cal), 6),
    'train_time': f'{rf_train_time:.0f}s',
}

print("\n  Results:")
for k, v in rf_results.items():
    print(f"    {k:<15}: {v}")
print("\nP12b complete ✓")

P12b — Random Forest with Target Encoding (RF-TE)
  RF subsample : 1,000,000 rows
  Severe rate  : 2.71%
  Features     : 34 (['t_year', 't_month']...+ ['s_street_te', 's_city_te'])

  Training time : 318s

  Results:
    model          : RF-TE
    features       : TE (34, 1M subsample)
    AUROC          : 0.7892
    AUPRC          : 0.1128
    ECE_uncal      : 0.184864
    ECE_cal        : 0.005551
    Brier_cal      : 0.027041
    train_time     : 318s

P12b complete ✓


---
## Step 5 — Summary Table + Save Results

In [6]:
# ── Print summary matching Table 4 columns ────────────────────────────────────
print("=" * 80)
print("SUMMARY — New rows for Table 4 (Model Comparison, US-Accidents 2023 test year)")
print("=" * 80)
print(f"{'Model':<22} {'AUROC':>8} {'AUPRC':>8} {'ECE(uncal)':>12} {'ECE(cal)':>10} {'Brier(cal)':>12}")
print("-" * 80)

# Reference rows (from paper Table 4)
refs = [
    ('Logistic Regression', 0.7081, 0.0584, 0.3327, 0.0055, 0.0279),
    ('Random Forest',       0.7686, 0.0838, 0.3480, 0.0059, 0.0275),
    ('XGBoost',            0.8215, 0.1306, 0.2394, 0.0065, 0.0266),
    ('LightGBM',           0.8193, 0.1311, 0.2226, 0.0065, 0.0267),
    ('MLP',                0.7607, 0.0720, 0.4082, 0.0045, 0.0276),
]
for name, auroc, auprc, ece_u, ece_c, brier in refs:
    print(f"  {name:<20} {auroc:>8.4f} {auprc:>8.4f} {ece_u:>12.4f} {ece_c:>10.4f} {brier:>12.4f}")

print("-" * 80)
for r in [cat_results, rf_results]:
    print(f"  {r['model']:<20} {r['AUROC']:>8.4f} {r['AUPRC']:>8.4f} "
          f"{r['ECE_uncal']:>12.4f} {r['ECE_cal']:>10.4f} {r['Brier_cal']:>12.4f}  ← NEW")

print(f"\n  No-skill AUPRC baseline: {no_skill:.4f}")

# ── Save JSON ─────────────────────────────────────────────────────────────────
results = {
    'catboost'        : cat_results,
    'rf_te'           : rf_results,
    'no_skill_auprc'  : no_skill,
    'reference_models': {
        'XGBoost' : {'AUROC': 0.8215, 'AUPRC': 0.1306, 'ECE_uncal': 0.2394, 'ECE_cal': 0.0065, 'Brier': 0.0266},
        'LightGBM': {'AUROC': 0.8193, 'AUPRC': 0.1311, 'ECE_uncal': 0.2226, 'ECE_cal': 0.0065, 'Brier': 0.0267},
    },
    'note'            : 'RF-TE trained on 1M stratified subsample. CatBoost on full training set.',
    'generated_at'    : str(datetime.now()),
}

out_path = LOGS_DIR / 'p12_sota_results.json'
with open(out_path, 'w') as f:
    json.dump(results, f, indent=2)
print(f"\nSaved to {out_path} ✓")
print("P12 complete ✓")

SUMMARY — New rows for Table 4 (Model Comparison, US-Accidents 2023 test year)
Model                     AUROC    AUPRC   ECE(uncal)   ECE(cal)   Brier(cal)
--------------------------------------------------------------------------------
  Logistic Regression    0.7081   0.0584       0.3327     0.0055       0.0279
  Random Forest          0.7686   0.0838       0.3480     0.0059       0.0275
  XGBoost                0.8215   0.1306       0.2394     0.0065       0.0266
  LightGBM               0.8193   0.1311       0.2226     0.0065       0.0267
  MLP                    0.7607   0.0720       0.4082     0.0045       0.0276
--------------------------------------------------------------------------------
  CatBoost               0.8140   0.1186       0.2665     0.0069       0.0269  ← NEW
  RF-TE                  0.7892   0.1128       0.1849     0.0056       0.0270  ← NEW

  No-skill AUPRC baseline: 0.0291

Saved to /content/drive/MyDrive/SAE-XCrash/logs/wp6/p12_sota_results.json ✓
P12 compl